# 📱 Smartphone Motion Lab (Google Colab)

**Real-Time Smartphone Sensor Data Streaming and Visualization in Colab**

This notebook reads **pitch**, **roll**, and **yaw** from a smartphone sensor stream (such as **Phyphox**) and displays the data in real time.

> **Important:** Google Colab cannot access local URLs like `192.168.x.x` directly. Use a **public URL** (for example, via ngrok) and paste it below.


## 1. Install Dependencies
Colab usually includes most of these, but this cell ensures everything is available.


In [ ]:
!pip -q install requests pandas matplotlib


## 2. Set the Public Sensor URL

Replace the placeholder URL with your public Phyphox endpoint, for example:

```python
URL = "https://abc123.ngrok-free.app/get?pitch&roll&yaw"
```


In [ ]:
URL = "https://your-public-url/get?pitch&roll&yaw"


## 3. Read One Sample
Run this first to verify that the connection works.


In [ ]:
import requests

def read_angles(url):
    try:
        r = requests.get(url, timeout=3)
        r.raise_for_status()
        data = r.json()
        pitch = data["buffer"]["pitch"]["buffer"][0]
        roll  = data["buffer"]["roll"]["buffer"][0]
        yaw   = data["buffer"]["yaw"]["buffer"][0]
        return float(pitch), float(roll), float(yaw)
    except Exception as e:
        print("Read error:", e)
        return None

sample = read_angles(URL)
print("Sample:", sample)


## 4. Real-Time Data Logging and Plotting
This cell reads a fixed number of samples, prints the latest values, shows the most recent rows, and updates a live plot.


In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

num_samples = 50      # change as needed
delay_s = 0.5         # seconds between reads

data_log = []

for i in range(num_samples):
    angles = read_angles(URL)

    if angles is None:
        print("Waiting for data...")
        time.sleep(1)
        continue

    pitch, roll, yaw = angles
    data_log.append({
        "sample": i + 1,
        "pitch": pitch,
        "roll": roll,
        "yaw": yaw,
    })

    df = pd.DataFrame(data_log)

    clear_output(wait=True)
    print(f"Latest -> Pitch: {pitch:.2f}°, Roll: {roll:.2f}°, Yaw: {yaw:.2f}°")
    display(df.tail())

    plt.figure(figsize=(10, 4))
    plt.plot(df["sample"], df["pitch"], label="Pitch")
    plt.plot(df["sample"], df["roll"], label="Roll")
    plt.plot(df["sample"], df["yaw"], label="Yaw")
    plt.xlabel("Sample")
    plt.ylabel("Angle (deg)")
    plt.title("Smartphone Motion Data")
    plt.legend()
    plt.grid(True)
    plt.show()

    time.sleep(delay_s)

print("Done.")


## 5. Save to CSV
Run this after logging data.


In [ ]:
if 'df' in globals() and not df.empty:
    df.to_csv("smartphone_motion_data.csv", index=False)
    print("Saved: smartphone_motion_data.csv")
else:
    print("No data available yet. Run the logging cell first.")


## 6. Optional: Download the CSV
Use this in Colab to download the saved file.


In [ ]:
from google.colab import files

try:
    files.download("smartphone_motion_data.csv")
except Exception as e:
    print("Download error:", e)
    print("Make sure the CSV file exists first.")


## 7. Troubleshooting
- Make sure the phone and Phyphox stream are running.
- Make sure the URL is **public**, not a local `192.168.x.x` address.
- If JSON parsing fails, print the raw response structure and adjust the field names.


In [ ]:
# Debug helper
try:
    raw = requests.get(URL, timeout=3).json()
    print(raw)
except Exception as e:
    print("Debug error:", e)
